# Code Execution Tool Calling - Simple Example

This notebook demonstrates **code execution tool calling**, where:
1. The model writes Python code that runs in Anthropic's sandbox
2. That code can call your custom tools as functions
3. Tool results go back to the **running code**, not to the model
4. Only the final output goes back to the model

This saves tokens by keeping intermediate data out of the model's context.

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key)
model = "claude-3-5-sonnet-20241022"

print("✓ Anthropic client initialized")

## Simple Tool: Add Numbers

A very simple tool that just adds two numbers.

In [ ]:
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    result = a + b
    print(f"Tool executed: add_numbers({a}, {b}) = {result}")
    return result

In [ ]:
# Define the tool for Claude
# Key difference: add "allowed_callers" to enable code execution to call this tool
tools = [
    {
        "type": "code_execution_20260120",
        "name": "code_execution"
    },
    {
        "name": "add_numbers",
        "description": "Add two numbers together and return the result. Use this when the user asks to add, sum, or perform addition on numbers. Do not use this for subtraction, multiplication, division, or other mathematical operations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                }
            },
            "required": ["a", "b"]
        },
        "allowed_callers": ["code_execution_20260120"]  # Only callable from code execution
    }
]

print("✓ Tool defined for code execution")

## Code Execution Tool Calling Loop

In code execution, the model writes Python that runs in a sandbox. When that code calls a tool, we intercept it, run it, and send the result back to the code (not the model).

In [ ]:
def code_execution_tool_call(user_message: str):
    """Use code execution - tool results go to the running code, not the model."""
    messages = [{"role": "user", "content": user_message}]
    
    print(f"\n📤 User: {user_message}")
    print("\n--- Code Execution Tool Calling Loop ---")
    
    while True:
        # Send message to Claude
        response = client.messages.create(
            model=model,
            max_tokens=256,
            tools=tools,
            messages=messages
        )
        
        print(f"\n[Stop reason: {response.stop_reason}]")
        
        # Check if we got tool use from code execution
        if response.stop_reason == "tool_use":
            tool_use = None
            for block in response.content:
                if block.type == "tool_use":
                    tool_use = block
                    break
            
            if tool_use:
                print(f"\n🔧 Code execution calls tool: {tool_use.name}")
                print(f"   Input: {tool_use.input}")
                print(f"   (Note: This is called FROM the sandbox code, not by the model)")
                
                # Execute the tool
                if tool_use.name == "add_numbers":
                    result = add_numbers(tool_use.input["a"], tool_use.input["b"])
                else:
                    result = f"Unknown tool: {tool_use.name}"
                
                # Add Claude's response to messages
                messages.append({
                    "role": "assistant",
                    "content": response.content
                })
                
                # Add tool result back to messages
                # Key difference: This goes to the running code, NOT the model's context
                messages.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": str(result)
                        }
                    ]
                })
                print(f"   Result sent to sandbox code (model never sees it)")
        else:
            # Code execution finished - extract text response
            final_response = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_response += block.text
            
            print(f"\n✅ Code execution output (what model saw): {final_response}")
            return final_response

## Test Code Execution Tool Calling

In [ ]:
# Simple request - model writes code to call add_numbers
code_execution_tool_call("Please add 15 and 27 for me.")

In [ ]:
# Request that benefits from code execution
# The model writes a script that calls add_numbers twice, aggregates the results, 
# and returns only the final answer
code_execution_tool_call("What is 10 + 20 and also 5 + 8? Give me both answers and their sum.")

## Key Characteristics of Code Execution Tool Calling

✓ **Fewer tokens**: Intermediate data stays out of the model's context  
✓ **Fewer round trips**: Loops and aggregation run in one execution  
✓ **Less model reasoning over data**: Arithmetic and filtering done by code  
✓ **Perfect for aggregation**: Model writes code to combine multiple tool calls  
✗ **Less visibility**: Can't see every tool call directly  
✗ **More complex**: Requires understanding the sandbox model  

**Best for**: Large datasets, queries requiring aggregation, repeated tool calls, when you want to minimize tokens

## Comparison: Direct vs Code Execution

| Aspect | Direct | Code Execution |
|--------|--------|----------------|
| **Who calls the tool** | Model decides | Model-written code |
| **Result goes to** | Model's context | Running code |
| **Tokens saved** | None | Yes (intermediate data) |
| **Number of API calls** | Multiple (one per tool use) | Fewer |
| **Visibility** | High (see every call) | Lower (see only final output) |
| **Control flow** | Model reasoning | Python code |

**Choose Direct when**: You need full control, the model must reason over all data, or working with small datasets.

**Choose Code Execution when**: You want to minimize tokens, aggregate results in code, or handle large datasets.